# Deep Learning for NLP

**Deep Learning** is a type of Machine Learning that uses **Neural Networks with multiple layers** to learn patterns from data.

Deep Learning can be used in NLP tasks such as:
- Text Classification
- Sentiment Analysis
- Question Answering

In this lab, we will build a **deep learning model** to classify Yelp restaurant reviews as:

- **0 = Negative**
- **1 = Positive**

<img src="https://drive.google.com/uc?export=view&id=1rUaLO1wEOcUZ10Kwd0nUvS3jQzrqoOcI" width="550">
<img src="https://drive.google.com/uc?export=view&id=1XKPTKwxmIpP5cZIkDajlai6aI9jUX-C8" width="400">

## 1. Load the Yelp Dataset

Each row contains:

- a restaurant review
- its sentiment label


In [1]:
import os
import pandas as pd

# The dataset ships next to the notebook, inside NLP_Lab6/dataset/
DATA = "dataset/07-yelp-dataset.txt"
if not os.path.exists(DATA):
    DATA = "07-yelp-dataset.txt"

df = pd.read_csv(DATA, sep="\t", names=["review", "label"])

print("Number of reviews:", len(df))
df.head()

Number of reviews: 1000


,review,label
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [2]:
print(df["label"].value_counts())

label
1    500
0    500
Name: count, dtype: int64


## 2. Preprocessing the Dataset

In [3]:
import re

def preprocess_text(text):

    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 3. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# NOTE: in the original notebook this line was indented inside preprocess_text(),
# after the return statement, so it never executed and every later cell worked on
# the raw, un-cleaned reviews. Moved out of the function so it actually runs.
df["review"] = df["review"].apply(preprocess_text)
df.head()

,review,label
0,wow loved this place,1
1,crust is not good,0
2,not tasty and the texture was just nasty,0
3,stopped by during the late may bank holiday of...,1
4,the selection on the menu was great and so wer...,1


## 3. Split the Data

We divide the dataset into:

- **80% training data** — used to train the model
- **20% testing data** — used to evaluate the model

`stratify=y` keeps the proportion of positive and negative reviews similar in both sets.

In [4]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["label"]

# stratify=y keeps the positive/negative ratio identical in both halves,
# which is what the markdown above describes.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))
print("train labels:", y_train.value_counts().to_dict())
print("test  labels:", y_test.value_counts().to_dict())

Training reviews: 800
Testing reviews: 200
train labels: {1: 400, 0: 400}
test  labels: {1: 100, 0: 100}


## 4. Convert Text to TF-IDF Vectors

A neural network cannot directly understand text.

**TF-IDF converts each review into a vector of numbers.**


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000, lowercase=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (800, 1000)
Testing TF-IDF shape: (200, 1000)


## 5. Convert the Data to PyTorch Tensors

PyTorch neural networks work with **tensors**.

In [6]:
import numpy as np
import torch
import torch.nn as nn

# Convert TF-IDF data to PyTorch tensors.
# np.array(..., dtype="float32") makes a fresh, writable copy. Handing torch a
# read-only NumPy buffer warns because the optimiser writes back into the same
# memory - and under pandas copy-on-write, y_train.values is read-only.
X_train_tensor = torch.from_numpy(np.array(X_train_tfidf.toarray(), dtype="float32"))
X_test_tensor = torch.from_numpy(np.array(X_test_tfidf.toarray(), dtype="float32"))

y_train_tensor = torch.from_numpy(np.array(y_train.values, dtype="float32").reshape(-1, 1))

## 6. Build a Simple Neural Network

Our network has:

1. **Input layer** — receives the TF-IDF features
2. **Hidden layer** — learns useful patterns
3. **ReLU activation** — adds non-linearity
4. **Output layer** — produces one value for binary classification

We do **not** put `Sigmoid` inside the model because `BCEWithLogitsLoss` applies it internally in a numerically stable way.

In [7]:
input_size = X_train_tensor.shape[1]
# torch.size([800, 1000])

model = nn.Sequential(
    nn.Linear(input_size, 64),   # Input layer
    nn.ReLU(),

    nn.Linear(64, 32),           # Hidden layer
    nn.ReLU(),

    nn.Linear(32, 1)             # Output layer
)

print(model)

Sequential(
  (0): Linear(in_features=1000, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)


## 7. Choose the Loss Function and Optimizer

- **BCEWithLogitsLoss** is used for binary classification.
To measure the model’s mistakes.
- **Adam** is responsible for changing the model's weights so that the loss becomes smaller.

In [8]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 8. Train the Neural Network

One **epoch** means the model has seen the full training dataset once.

For every epoch:

1. Make predictions
2. Calculate the loss
3. Calculate gradients with backpropagation
4. Update the weights

In [9]:
epochs = 10

for epoch in range(epochs):
    model.train()

    # 1. Forward pass
    outputs = model(X_train_tensor)

    # 2. Calculate loss
    loss = loss_function(outputs, y_train_tensor)

    # 3. Backpropagation
    loss.backward()

    # 4. Update model weights
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

Epoch 5/10 - Loss: 0.6417
Epoch 10/10 - Loss: 0.4160


## 9. Evaluate the Model

The model outputs raw scores called **logits**.

We apply `sigmoid` to convert them to probabilities between 0 and 1.

- Probability **≥ 0.5 → Positive (1)**
- Probability **< 0.5 → Negative (0)**

In [10]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

with torch.no_grad():
    test = model(X_test_tensor)
    test_probabilities = torch.sigmoid(test)
    predictions = (test_probabilities >= 0.5).int().numpy().flatten()

accuracy = accuracy_score(y_test, predictions)

print(f"Test Accuracy: {accuracy:.3f}")

Test Accuracy: 0.775


## 10. Classification Report and Confusion Matrix

The classification report shows:

- **Precision**
- **Recall**
- **F1-score**

The confusion matrix shows how many reviews were classified correctly and incorrectly.

In [11]:
print("Classification Report:")
print(classification_report(y_test, predictions, target_names=["Negative", "Positive"]))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

Classification Report:
              precision    recall  f1-score   support

    Negative       0.75      0.83      0.79       100
    Positive       0.81      0.72      0.76       100

    accuracy                           0.78       200
   macro avg       0.78      0.77      0.77       200
weighted avg       0.78      0.78      0.77       200

Confusion Matrix:
[[83 17]
 [28 72]]


## Task 1 – Represent Reviews Using Word2Vec

In the previous part of the lab, TF-IDF was used to convert each review into numerical features.
Now, use Word2Vec instead.

Requirements:
Tokenize the training and testing reviews.
Train a Word2Vec model using the training reviews.
Use:
vector_size = 200
window = 5
min_count = 1
sg = 1

Convert each review into a single vector.
Convert the resulting vectors into PyTorch tensors.


In [12]:
import nltk
for _pkg in ("punkt", "punkt_tab"):
    nltk.download(_pkg, quiet=True)

import numpy as np
import gensim
import torch
from nltk.tokenize import word_tokenize

# ---- Task 1: represent reviews with Word2Vec --------------------------
# 1. tokenize the training and testing reviews
train_tokens = [word_tokenize(t) for t in X_train]
test_tokens = [word_tokenize(t) for t in X_test]

# 2. train a skip-gram model on the TRAINING reviews only.
#    Training on the test reviews as well would leak their vocabulary into
#    the embeddings used to predict them.
w2v = gensim.models.Word2Vec(
    train_tokens, vector_size=200, window=5, min_count=1, sg=1
)
print("vocabulary size:", len(w2v.wv))
print("vector for 'good':", w2v.wv["good"][:5].round(3), "...")

# 3. one vector per review. Word2Vec gives a vector per word, but the network
#    needs a fixed-size vector per review, so average the word vectors. This is
#    the standard trick and it is what "convert each review into a single
#    vector" means here.
VECTOR_SIZE = 200

def review_vector(token_list):
    vectors = [w2v.wv[w] for w in token_list if w in w2v.wv]
    if not vectors:
        return np.zeros(VECTOR_SIZE, dtype="float32")
    return np.mean(vectors, axis=0)

X_train_w2v = np.vstack([review_vector(t) for t in train_tokens]).astype("float32")
X_test_w2v = np.vstack([review_vector(t) for t in test_tokens]).astype("float32")

# 4. to PyTorch tensors (same shape convention as the TF-IDF version).
#    torch.tensor() always copies, which matters: gensim's vectors and pandas'
#    .values are read-only, and handing torch a read-only buffer warns because
#    the optimiser writes back into the same memory.
X_train_tensor = torch.tensor(X_train_w2v, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_w2v, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values.reshape(-1, 1), dtype=torch.float32)

print("train tensor:", tuple(X_train_tensor.shape))
print("test  tensor:", tuple(X_test_tensor.shape))
print()

# ---- already worth checking here: is there enough text for these embeddings?
n_tokens = sum(len(t) for t in train_tokens)
n_vocab = len(w2v.wv)
print(f"training text : {n_tokens:,} tokens")
print(f"vocabulary    : {n_vocab:,} words")
print(f"embedding     : {n_vocab * VECTOR_SIZE:,} parameters")
print(f"              -> {n_tokens / (n_vocab * VECTOR_SIZE):.3f} tokens per parameter")
print("    (for comparison, word2vec is normally trained on billions of tokens)")
print()

# ... and how alike are the review vectors themselves?
unit = X_train_w2v / (np.linalg.norm(X_train_w2v, axis=1, keepdims=True) + 1e-9)
cos_to_first = unit @ unit[0]
print("cosine similarity of each training review vector to review #0:")
print(f"   mean {cos_to_first.mean():.4f}   min {cos_to_first.min():.4f}   "
      f"max {cos_to_first.max():.4f}")
print("   1.0 would mean every review has the same vector.")

vocabulary size: 1786
vector for 'good': [ 0.014 -0.034  0.014  0.04   0.162] ...
train tensor: (800, 200)
test  tensor: (200, 200)

training text : 8,622 tokens
vocabulary    : 1,786 words
embedding     : 357,200 parameters
              -> 0.024 tokens per parameter
    (for comparison, word2vec is normally trained on billions of tokens)

cosine similarity of each training review vector to review #0:
   mean 0.9998   min 0.9935   max 1.0000
   1.0 would mean every review has the same vector.


Note on the numbers here: 8,622 training tokens against a 1,786-word vocabulary is a very small corpus for word2vec. Keep an eye on the mean cosine printed above as you go into Task 2 — if the review vectors all point the same way, no classifier downstream can separate them.

## Task 2 – Build a Different Neural Network

Use the Word2Vec vectors from Task 1 as the input to a new neural network.
Your network must contain:
An input layer
Three hidden layers
ReLU activation functions
One output neuron
Use the following architecture:

Input
   
   ↓

128 neurons
   
   ↓

64 neurons
   
   ↓

32 neurons
   
   ↓

1 output neuron

Train the model using:
, BCEWithLogitsLoss
, Adam optimizer
, Learning rate = 0.001
, 15 epochs.
Then calculate the test accuracy.

In [13]:
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ---- Task 2: a deeper network on the Word2Vec features ----------------
input_size = X_train_tensor.shape[1]      # 200, one averaged embedding per review

model2 = nn.Sequential(
    nn.Linear(input_size, 128),   # input layer
    nn.ReLU(),
    nn.Linear(128, 64),           # hidden 1
    nn.ReLU(),
    nn.Linear(64, 32),            # hidden 2
    nn.ReLU(),
    nn.Linear(32, 1),             # output neuron
)
print(model2)
print()

loss_function2 = nn.BCEWithLogitsLoss()
optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.001)

# Exactly what the task asks for: BCEWithLogitsLoss, Adam at lr=0.001, 15 epochs.
# Note this is full-batch training, so 15 epochs = 15 weight updates, and that
# number matters - the next cell comes back to it.
epochs = 15
for epoch in range(epochs):
    model2.train()
    optimizer2.zero_grad()                 # clear last epoch's gradients

    outputs = model2(X_train_tensor)       # forward
    loss = loss_function2(outputs, y_train_tensor)
    loss.backward()                        # backward
    optimizer2.step()                      # update

    if (epoch + 1) % 3 == 0:
        print(f"Epoch {epoch+1:2}/{epochs} - Loss: {loss.item():.4f}")

# test accuracy
model2.eval()
with torch.no_grad():
    logits = model2(X_test_tensor)
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities >= 0.5).int().numpy().flatten()

print()
print(f"Test Accuracy: {accuracy_score(y_test, predictions):.3f}")
print()
print(classification_report(y_test, predictions, target_names=["Negative", "Positive"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))
print()
print("The loss is sitting on ln(2) = 0.693 and the model predicts one class")
print("for everything, so the accuracy is at chance. The next cell works out why.")

Sequential(
  (0): Linear(in_features=200, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=1, bias=True)
)

Epoch  3/15 - Loss: 0.6933
Epoch  6/15 - Loss: 0.6932
Epoch  9/15 - Loss: 0.6932
Epoch 12/15 - Loss: 0.6932
Epoch 15/15 - Loss: 0.6931

Test Accuracy: 0.495

              precision    recall  f1-score   support

    Negative       0.50      0.99      0.66       100
    Positive       0.00      0.00      0.00       100

    accuracy                           0.49       200
   macro avg       0.25      0.49      0.33       200
weighted avg       0.25      0.49      0.33       200

Confusion Matrix:
[[ 99   1]
 [100   0]]

The loss is sitting on ln(2) = 0.693 and the model predicts one class
for everything, so the accuracy is at chance. The next cell works out why.


In [14]:
# ---- So why is that at chance level? --------------------------------------
# The guess you would reach for first is "not enough training": the loop above
# is full-batch, so 15 epochs is only 15 gradient steps at lr=0.001. Test it.

import copy
from sklearn.metrics import accuracy_score

torch.manual_seed(0)
probe = nn.Sequential(
    nn.Linear(X_train_tensor.shape[1], 128), nn.ReLU(),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 1),
)
opt_probe = torch.optim.Adam(probe.parameters(), lr=0.001)
loss_probe = nn.BCEWithLogitsLoss()

n = len(X_train_tensor)
print("mini-batches of 32 -> 25 updates per epoch instead of 1:")
for epoch in range(15):
    perm = torch.randperm(n)
    for i in range(0, n, 32):
        idx = perm[i:i + 32]
        probe.train(); opt_probe.zero_grad()
        loss_probe(probe(X_train_tensor[idx]), y_train_tensor[idx]).backward()
        opt_probe.step()

probe.eval()
with torch.no_grad():
    preds = (torch.sigmoid(probe(X_test_tensor)) >= 0.5).int().numpy().flatten()
print(f"   -> accuracy {accuracy_score(y_test, preds):.3f}   (375 updates, not 15)")
print()

# More updates did not help, so training is not the bottleneck. Now hold the
# network, the loss, the optimizer, the learning rate and the epoch count
# completely fixed and change only the input features: TF-IDF instead of the
# averaged Word2Vec vectors.
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_control = TfidfVectorizer(max_features=1000)
A = np.ascontiguousarray(tfidf_control.fit_transform(X_train).toarray(), dtype="float32")
B = np.ascontiguousarray(tfidf_control.transform(X_test).toarray(), dtype="float32")

torch.manual_seed(0)
control = nn.Sequential(
    nn.Linear(A.shape[1], 128), nn.ReLU(),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 1),
)
opt_c = torch.optim.Adam(control.parameters(), lr=0.001)
loss_c = nn.BCEWithLogitsLoss()
ytr_t = torch.tensor(y_train.values.reshape(-1, 1), dtype=torch.float32)
A_t, B_t = torch.tensor(A, dtype=torch.float32), torch.tensor(B, dtype=torch.float32)

for epoch in range(15):
    perm = torch.randperm(len(A_t))
    for i in range(0, len(A_t), 32):
        idx = perm[i:i + 32]
        control.train(); opt_c.zero_grad()
        loss_c(control(A_t[idx]), ytr_t[idx]).backward()
        opt_c.step()

control.eval()
with torch.no_grad():
    preds_c = (torch.sigmoid(control(B_t)) >= 0.5).int().numpy().flatten()

print("same network, same loss, same optimizer, same lr, same 15 epochs,")
print("only the input features changed:")
print(f"   averaged Word2Vec (200 dims) -> {accuracy_score(y_test, preds):.3f}")
print(f"   TF-IDF           (1000 dims) -> {accuracy_score(y_test, preds_c):.3f}")
print()
print("So the network and the training loop are fine. The embeddings are the")
print("problem: they were fit to 8,622 tokens, which is about 0.02 training")
print("tokens per embedding parameter. They are close to random, and averaging")
print("about eleven of them leaves a vector pointing in almost the same")
print("direction for every review - the mean cosine of 0.9998 from Task 1.")
print()
print("Fixes, in order of how much they help here:")
print("  1. train the embeddings on a large unlabelled corpus first (millions")
print("     of tokens), then average them for this task - this is what")
print("     pretrained word vectors are for;")
print("  2. for a dataset this small, skip the embedding and use TF-IDF - it")
print("     needs no training and is 30 points more accurate;")
print("  3. do not mean-pool. Averaging throws away which words were present,")
print("     which is exactly the information a bag-of-words task needs.")

mini-batches of 32 -> 25 updates per epoch instead of 1:


   -> accuracy 0.500   (375 updates, not 15)



same network, same loss, same optimizer, same lr, same 15 epochs,
only the input features changed:
   averaged Word2Vec (200 dims) -> 0.500
   TF-IDF           (1000 dims) -> 0.815

So the network and the training loop are fine. The embeddings are the
problem: they were fit to 8,622 tokens, which is about 0.02 training
tokens per embedding parameter. They are close to random, and averaging
about eleven of them leaves a vector pointing in almost the same
direction for every review - the mean cosine of 0.9998 from Task 1.

Fixes, in order of how much they help here:
  1. train the embeddings on a large unlabelled corpus first (millions
     of tokens), then average them for this task - this is what
     pretrained word vectors are for;
  2. for a dataset this small, skip the embedding and use TF-IDF - it
     needs no training and is 30 points more accurate;
  3. do not mean-pool. Averaging throws away which words were present,
     which is exactly the information a bag-of-words task ne

**Correcting the markdown above**

Two things in the lab text do not match the code, and both were fixed:

- Section 2: `df["review"] = df["review"].apply(preprocess_text)` was indented *inside* the function, after the `return`. It never ran, so every later cell worked on raw, un-cleaned review text. It has been moved out of the function.
- Section 3: the markdown says `stratify=y` keeps the positive/negative ratio identical in both halves, but the call did not pass it. It does now.

**Result of Task 1 and Task 2**

Task 1 produces the tensors and Task 2 trains exactly the network and settings the lab specifies, and the honest answer is that it does **not** work: the loss stays on ln(2) and the test accuracy is about 0.50 — the same as always predicting the majority class. The next cell shows it is the embeddings, not the network, and demonstrates the fix. This is the useful part of the lab: a working training loop on a representation that carries no signal still scores at chance, and the metric alone will not tell you which of those two you are looking at.